# Final node embedding: clustering and cell content

**Role:** Analysis.

Load any supported saved GIN or FiLM checkpoint and study how its final node representation partitions cells. The notebook separates model inputs from appended globals, fits the atlas on training cells, orders clusters by predicted curvature, and reports validation marker composition and curvature. All cells are included unless the visible sampling settings are changed. It saves the atlas and annotated validation table; no model or target transform is fitted here.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Select a saved model

Select one checkpoint/fold; independent models have unrelated embedding coordinates and are not concatenated. Both ordinary GIN and FiLM are supported, including full or exclusive fates. By default every validation cell is included, so cluster marker fractions describe center cells rather than selected LGR5 neighborhoods.

In [ ]:
import json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.artifacts.runs import AnalysisRun
from src.analysis.metrics.evaluation import prediction_table
from src.artifacts.paths import resolve_training_run
TRAINING_NOTEBOOK = 'gin_depth_training'  # Folder under training_results containing the saved run.
TRAINING_RUN = None  # Set a run directory name; None selects only if exactly one exists.
RUN_DIR = resolve_training_run(ROOT, TRAINING_NOTEBOOK, TRAINING_RUN)  # Also accepts an explicit historical path.
run = AnalysisRun(RUN_DIR)
display(run.records)

# Saved model selection
MODEL_KEY = run.records.iloc[0].key  # Checkpoint key selected from the saved model catalog.

# Inference resources
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.
selection = run.select(MODEL_KEY, device=DEVICE)
model = selection['model']
marker_names = selection['marker_names']
g_train, g_val = selection['groups']['train'], selection['groups']['val']
OUTPUT_DIR = RUN_DIR/'analysis'/'clustering'/MODEL_KEY  # Directory for this analysis’s tables and figures.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Extract final node representations

Extract representations before global features are concatenated to the prediction head. FiLM conditioning remains part of the representation. Fit a scaler, PCA and GMM on training cells; assign validation cells without fitting to their targets. Cluster numbers are then ordered by mean predicted physical curvature on training cells.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.manifold import TSNE
from src.analysis.embeddings.clustering import extract_node_embeddings

# Embedding and clustering parameters
N_CLUSTERS = 6  # Number of embedding clusters to fit.
PCA_DIM = 16  # PCA dimensions used before clustering; None keeps all.

# Embedding sampling
MAX_FIT_CELLS = 30000  # Maximum training cells used to fit the embedding atlas.
MAX_PLOT_CELLS = 5000  # Maximum validation cells shown in scatter plots.

# Ablation sampling
SEED = 42  # Random seed for reproducible sampling or initialization.
train_repr = extract_node_embeddings(g_train, model, device=DEVICE)
val_repr = extract_node_embeddings(g_val, model, device=DEVICE)
rng = np.random.default_rng(SEED)
fit_ids = np.sort(rng.choice(len(train_repr.embeddings_local), min(MAX_FIT_CELLS,len(train_repr.embeddings_local)), replace=False))
scaler = StandardScaler().fit(train_repr.embeddings_local[fit_ids])
pca = PCA(n_components=min(PCA_DIM, len(fit_ids), train_repr.embeddings_local.shape[1]), random_state=SEED)
train_z = pca.fit_transform(scaler.transform(train_repr.embeddings_local[fit_ids]))
val_z = pca.transform(scaler.transform(val_repr.embeddings_local))
gmm = GaussianMixture(N_CLUSTERS, random_state=SEED, reg_covar=1e-5).fit(train_z)
training_predictions = prediction_table(selection, role='train', device=DEVICE)
raw_labels = gmm.predict(train_z)
order = pd.DataFrame({'cluster':raw_labels,'curvature':training_predictions.y_pred.to_numpy()[fit_ids]}).groupby('cluster').curvature.mean().sort_values().index.tolist()
order += [c for c in range(N_CLUSTERS) if c not in order]
label_map = {c:i for i,c in enumerate(order)}
labels = np.array([label_map[c] for c in gmm.predict(val_z)])
predictions = prediction_table(selection, device=DEVICE).assign(cluster=labels)
display(predictions.groupby('cluster').agg(cells=('node','size'),organoids=('organoid_str','nunique'),measured_curvature=('y_true','mean'),predicted_curvature=('y_pred','mean')))
print('PCA explained variance:', pca.explained_variance_ratio_.sum())

## Cluster maps and cell content

PCA and t-SNE show the same sampled validation cells and cluster labels. PCA distances are in standardized representation coordinates; t-SNE is a visualization only. Marker proportions use each cell’s own measured fate. Unassigned is shown separately and coexpression is retained in nonexclusive datasets.

In [ ]:
plot_ids = np.sort(rng.choice(len(val_z), min(MAX_PLOT_CELLS,len(val_z)), replace=False))
xy = TSNE(n_components=2, perplexity=min(30.,max(1.,(len(plot_ids)-1)/3)), random_state=SEED, init='pca').fit_transform(val_z[plot_ids])
fig, axes = plt.subplots(1, 3, figsize=(16,4))
axes[0].scatter(val_z[plot_ids,0],val_z[plot_ids,1] if val_z.shape[1]>1 else np.zeros(len(plot_ids)),c=labels[plot_ids],s=3,cmap='tab10')
axes[0].set(xlabel='PC1',ylabel='PC2',title='Final node embedding')
axes[1].scatter(xy[:,0],xy[:,1],c=labels[plot_ids],s=3,cmap='tab10'); axes[1].set_title('t-SNE: same cluster assignment')
fates = val_repr.x_markers[:,:len(marker_names)]
content = pd.DataFrame(np.c_[fates,(fates.sum(1)==0)],columns=[*marker_names,'Unassigned']).assign(cluster=labels).groupby('cluster').mean()
im=axes[2].imshow(content,aspect='auto',vmin=0,vmax=1)
axes[2].set(xticks=range(len(content.columns)),xticklabels=content.columns,yticks=range(N_CLUSTERS),ylabel='Cluster (curvature ordered)')
axes[2].tick_params(axis='x',rotation=90); fig.colorbar(im,ax=axes[2],label='Fraction of cluster cells')
plt.tight_layout();fig.savefig(OUTPUT_DIR/'clusters.png',dpi=180);plt.show()
content.to_csv(OUTPUT_DIR/'marker_content.csv'); predictions.to_csv(OUTPUT_DIR/'cells.csv.gz',index=False)
import pickle
with (OUTPUT_DIR/'atlas.pkl').open('wb') as handle:
    pickle.dump(dict(scaler=scaler,pca=pca,gmm=gmm,label_map=label_map,model_key=MODEL_KEY),handle)